# Сравнение моделей: EDA, SVM, Fuzzy k-NN, AutoML

Датасет `dispensarization_data_2026.csv`, цель `ССЗ_риск_высокий`.  
Логика совпадает с `run_dispensary_experiments.py`.

In [ ]:
# импорты и пути
import json
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.metrics import RocCurveDisplay, auc, confusion_matrix, roc_curve
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from run_dispensary_experiments import (
    AUTOML_TIME_BUDGET,
    DATA_PATH,
    FuzzyKNNClassifier,
    RANDOM_STATE,
    RESULTS_PATH,
    eda_summary,
    evaluate,
    impute_train_test,
    load_and_prepare,
    run_automl_frameworks,
)

warnings.filterwarnings("ignore")

ROOT = Path(".").resolve()
IMAGES_DIR = ROOT / "report_assets" / "images"
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"font.size": 11, "figure.dpi": 120})
plt.style.use("ggplot")
sns.set_theme(style="whitegrid")

print(f"Корень: {ROOT}")
print(f"Данные: {DATA_PATH}")

## 1. Загрузка данных и EDA

In [ ]:
df, X, y = load_and_prepare()
eda = eda_summary(df, X, y)
TARGET = eda["target"]

print(f"Строк: {eda['n_rows']}, признаков: {eda['n_features']}")
print(f"Класс 1: {eda['class_1']} ({eda['class_1_pct']}%)")
print(f"Пропусков (ячеек): {eda['missing_total']}")
display(pd.DataFrame(X.describe().round(2)))

In [ ]:
# баланс классов и пропуски
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
counts = df[TARGET].value_counts().sort_index()
axes[0].bar(["Нет риска (0)", "Высокий риск (1)"], counts.values, color=["#7BAFD4", "#2253A1"])
axes[0].set_title(f"Баланс классов: {TARGET}")
axes[0].set_ylabel("Пациентов")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 8, str(v), ha="center")

miss = df.isna().mean().sort_values(ascending=False)
miss = (miss[miss > 0].head(8) * 100)
axes[1].barh(miss.index, miss.values, color="#0E2D69", alpha=0.85)
axes[1].set_xlabel("Пропуски, %")
axes[1].set_title("Топ пропусков")
plt.tight_layout()
plt.savefig(IMAGES_DIR / "eda_class_missing.png", bbox_inches="tight")
plt.show()

In [ ]:
# возраст и корреляции
fig, ax = plt.subplots(figsize=(8, 3.5))
df["Возраст"].hist(bins=20, color="#2253A1", edgecolor="white", ax=ax)
ax.set_title("Возраст (40–65)")
ax.set_xlabel("Возраст")
plt.tight_layout()
plt.savefig(IMAGES_DIR / "eda_age_distribution.png", bbox_inches="tight")
plt.show()

num = df.select_dtypes(include=[np.number]).drop(
    columns=[c for c in [TARGET, "Статус_глюкозы", "Доклинический_риск"] if c in df.columns],
    errors="ignore",
)
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(num.corr(), cmap="RdBu_r", center=0, ax=ax, annot=False)
ax.set_title("Корреляции признаков")
plt.tight_layout()
plt.savefig(IMAGES_DIR / "eda_correlation_heatmap.png", bbox_inches="tight")
plt.show()

## 2. Разбиение train/test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
X_train_imp, X_test_imp = impute_train_test(X_train, X_test)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

scaler_steps = [
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
]
print(f"train: {len(X_train)}, test: {len(X_test)}")

## 3. SVM и Fuzzy k-NN (GridSearchCV, scoring=roc_auc)

In [ ]:
svm_grid = GridSearchCV(
    Pipeline(scaler_steps + [("clf", SVC(
        kernel="rbf", class_weight="balanced", probability=True, random_state=RANDOM_STATE
    ))]),
    {"clf__C": [0.1, 1, 10, 100], "clf__gamma": ["scale", 0.01, 0.1]},
    cv=cv, scoring="roc_auc", n_jobs=-1,
)

fuzzy_grid = GridSearchCV(
    Pipeline(scaler_steps + [("clf", FuzzyKNNClassifier())]),
    {"clf__k": [5, 7, 11, 15], "clf__m": [1.5, 2.0, 2.5]},
    cv=cv, scoring="roc_auc", n_jobs=1,
)

print("SVM...")
svm_result = evaluate("SVM (GridSearch)", svm_grid, X_train, X_test, y_train, y_test)
print("Fuzzy k-NN...")
fuzzy_result = evaluate("Fuzzy k-NN (GridSearch)", fuzzy_grid, X_train, X_test, y_train, y_test)

svm_best = svm_grid.best_estimator_
fuzzy_best = fuzzy_grid.best_estimator_
print(svm_result)
print(fuzzy_result)

## 4. AutoML: FLAML, H2O, MLJAR, AutoGluon

In [ ]:
print(f"AutoML, бюджет {AUTOML_TIME_BUDGET} с на фреймворк...")
t0 = time.perf_counter()
automl_results = run_automl_frameworks(X_train_imp, X_test_imp, y_train, y_test)
automl_total_sec = round(time.perf_counter() - t0, 2)

completed = [r for r in automl_results if not r.get("skipped")]
best_automl = max(completed, key=lambda r: r["roc_auc"]) if completed else None
flaml_proba = flaml_pred = None

# для ROC — переобучим FLAML короче, если был пропуск
if best_automl and best_automl["model"] == "AutoML/FLAML":
    try:
        from flaml import AutoML
        am = AutoML()
        am.fit(X_train_imp, y_train, task="classification", time_budget=60, metric="roc_auc", seed=RANDOM_STATE)
        flaml_proba = am.predict_proba(X_test_imp)[:, 1]
        flaml_pred = am.predict(X_test_imp)
    except Exception as e:
        print("FLAML для ROC:", e)

## 5. Сводка результатов и гипотеза

In [ ]:
if best_automl:
    automl_block = {**best_automl, "model": f"AutoML (лучший: {best_automl['model']})", "automl_total_sec": automl_total_sec, "automl_all": automl_results}
else:
    automl_block = {"model": "AutoML (нет)", "roc_auc": 0, "f1": 0, "balanced_accuracy": 0, "automl_total_sec": automl_total_sec, "automl_all": automl_results}

results = {
    "dataset": "data/dispensarization_data_2026.csv",
    "task": "Прогноз высокого ССЗ-риска (ССЗ_риск_высокий), пациенты 40–65 лет",
    "split": "train/test 80/20, stratified, random_state=42",
    "cv": "StratifiedKFold 5 на train для SVM и Fuzzy k-NN",
    "eda": eda,
    "models": [svm_result, fuzzy_result, automl_block],
}
svm_auc, best_auc = svm_result["roc_auc"], automl_block.get("roc_auc") or 0
results["hypothesis"] = {
    "automl_auc_gain_pct": round(100 * (best_auc - svm_auc) / svm_auc, 2) if svm_auc and best_auc else 0,
    "confirmed_partially": (best_auc - svm_auc) < 0.05 if best_auc else True,
}

RESULTS_PATH.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Сохранено: {RESULTS_PATH}")
print(f"Гипотеза: прирост AUC {results['hypothesis']['automl_auc_gain_pct']}%")

In [ ]:
# таблица сравнения
rows = []
for m in results["models"][:2]:
    rows.append({"Модель": m["model"], "ROC-AUC": m["roc_auc"], "F1": m["f1"],
                 "Balanced acc.": m["balanced_accuracy"], "Fit, с": m["fit_time_sec"]})
for m in automl_results:
    if m.get("skipped"):
        rows.append({"Модель": m["model"], "ROC-AUC": "—", "F1": "—", "Balanced acc.": "—", "Fit, с": "пропуск"})
    else:
        rows.append({"Модель": m["model"], "ROC-AUC": m["roc_auc"], "F1": m["f1"],
                     "Balanced acc.": m["balanced_accuracy"], "Fit, с": m["fit_time_sec"]})
df_cmp = pd.DataFrame(rows)
display(df_cmp)

## 6. Графики сравнения моделей

In [ ]:
# ROC-AUC и F1 — три основные модели
plot_df = df_cmp[df_cmp["ROC-AUC"] != "—"].copy()
plot_df["ROC-AUC"] = plot_df["ROC-AUC"].astype(float)
plot_df["F1"] = plot_df["F1"].astype(float)
main = plot_df[plot_df["Модель"].str.contains("SVM|Fuzzy|FLAML", regex=True)].head(3)
if len(main) < 3 and best_automl:
    main = plot_df.head(3)

labels = main["Модель"].str.replace("AutoML/", "").str.replace(" (GridSearch)", "").tolist()
x = np.arange(len(labels))
w = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - w/2, main["ROC-AUC"], w, label="ROC-AUC", color="#2253A1")
ax.bar(x + w/2, main["F1"], w, label="F1", color="#0E2D69", alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=12, ha="right")
ax.set_ylim(0, 1.05)
ax.set_title("Метрики на test")
ax.legend()
plt.tight_layout()
plt.savefig(IMAGES_DIR / "metrics_auc_f1.png", bbox_inches="tight")
plt.show()

In [ ]:
# ROC-AUC всех моделей
auc_df = plot_df.copy()
fig, ax = plt.subplots(figsize=(9, max(3, len(auc_df) * 0.5)))
colors = ["#2253A1" if "SVM" in m else "#5B8FD6" if "Fuzzy" in m else "#7BAFD4" for m in auc_df["Модель"]]
ax.barh(auc_df["Модель"], auc_df["ROC-AUC"], color=colors)
ax.set_xlim(0.85, 1.01)
ax.set_xlabel("ROC-AUC")
ax.set_title("ROC-AUC: все модели")
plt.tight_layout()
plt.savefig(IMAGES_DIR / "metrics_roc_auc_all.png", bbox_inches="tight")
plt.show()

In [ ]:
# время обучения
time_df = plot_df.copy()
time_df["Fit, с"] = pd.to_numeric(time_df["Fit, с"], errors="coerce")
time_df = time_df.dropna(subset=["Fit, с"])
fig, ax = plt.subplots(figsize=(9, max(3, len(time_df) * 0.45)))
ax.barh(time_df["Модель"], time_df["Fit, с"], color="#2253A1", alpha=0.85)
ax.set_xlabel("Секунды")
ax.set_title(f"Время обучения (AutoML цикл: {automl_total_sec} с)")
plt.tight_layout()
plt.savefig(IMAGES_DIR / "metrics_fit_time.png", bbox_inches="tight")
plt.show()

In [ ]:
# AutoML-фреймворки (если запускались)
ok = [m for m in automl_results if not m.get("skipped")]
if ok:
    names = [m["model"].replace("AutoML/", "") for m in ok]
    fig, ax = plt.subplots(figsize=(7, 4))
    x = np.arange(len(names))
    w = 0.35
    ax.bar(x - w/2, [m["roc_auc"] for m in ok], w, label="ROC-AUC")
    ax.bar(x + w/2, [m["f1"] for m in ok], w, label="F1")
    ax.set_xticks(x)
    ax.set_xticklabels(names)
    ax.set_ylim(0, 1.05)
    ax.set_title("AutoML-фреймворки (test)")
    ax.legend()
    plt.tight_layout()
    plt.savefig(IMAGES_DIR / "automl_frameworks.png", bbox_inches="tight")
    plt.show()
else:
    print("Нет успешных AutoML-прогонов для графика")

In [ ]:
# ROC-кривые
fig, ax = plt.subplots(figsize=(7, 6))
colors = {"SVM": "#2253A1", "Fuzzy k-NN": "#5B8FD6", "FLAML": "#0E2D69"}
RocCurveDisplay.from_estimator(svm_best, X_test, y_test, ax=ax, name="SVM", color=colors["SVM"])
RocCurveDisplay.from_estimator(fuzzy_best, X_test, y_test, ax=ax, name="Fuzzy k-NN", color=colors["Fuzzy k-NN"])
if flaml_proba is not None:
    fpr, tpr, _ = roc_curve(y_test, flaml_proba)
    ax.plot(fpr, tpr, color=colors["FLAML"], lw=2, label=f"FLAML (AUC={auc(fpr, tpr):.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=0.8, alpha=0.5)
ax.set_title("ROC-кривые (test)")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(IMAGES_DIR / "roc_curves.png", bbox_inches="tight")
plt.show()

In [ ]:
# матрицы ошибок
panels = [("SVM", svm_best), ("Fuzzy k-NN", fuzzy_best)]
if flaml_pred is not None:
    panels.append(("FLAML", None))

fig, axes = plt.subplots(1, len(panels), figsize=(4 * len(panels), 4))
if len(panels) == 1:
    axes = [axes]
for ax, (name, model) in zip(axes, panels):
    pred = flaml_pred if name == "FLAML" else model.predict(X_test)
    cm = confusion_matrix(y_test, pred)
    ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center", color="white", fontweight="bold")
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_title(name)
    ax.set_xlabel("Предсказание")
    ax.set_ylabel("Истина")
fig.suptitle("Confusion matrix (test)", y=1.02)
plt.tight_layout()
plt.savefig(IMAGES_DIR / "confusion_matrices.png", bbox_inches="tight")
plt.show()

print(f"Графики: {IMAGES_DIR}")